# Lab 11: Flag Accounts for Credit Review

You are the analyst at **Summit Gear Co.** Last week you predicted how long each account takes
to pay. This week the credit team asks a different question about the same 2,200 accounts:
which ones are likely to default, so a credit analyst can review them before the next order
ships on terms? A review costs about $400 of analyst time. A default the team does not see
coming costs about $2,000 after recoveries. An AI tool can write the fitting code. You choose
the predictors, check that each fit ran, and recommend a cutoff to the credit team with what
it costs.

> **First:** in Colab, choose **File > Save a copy in Drive** and work in your copy. The
> copy that opens from the link does not save your work.

> **Then:** **Runtime > Run all.** The notebook runs clean before you have written
> anything. The **Final check** at the bottom lists what is still yours to do.

Plan on about 3 hours. The [Lab 11 page](https://seanmccaman.com/acctg6155/2026-fall/week-11/lab/lab_11.html) has the AI tutor, the data dictionary,
what each written cell is graded on, and what to submit.

## How this lab works

1. **You specify.** The target, the predictors you chose, and why each one belongs.
2. **The AI writes the fit.** Ask Gemini in Colab, or any AI tool, for the `statsmodels` code, and paste it into the cell marked TODO.
3. **You validate it ran.** The right rows, a model that converged, and the right predictors. The cell under each fit prints all three.
4. **You interpret.** What the numbers mean for the credit team, in your own words.

Keep the names the TODO cells give you (`PREDICTORS`, `logit_full`, `logit_train`,
`train_prob`, `test_prob`, `test_auc`, `train_auc`) and the three `def` lines exactly as
given. The Final check reads them by name.

## Section 0: Setup and load

Colab already has `pandas`, `statsmodels` and `scikit-learn`. On your own machine, run
`pip install pandas statsmodels scikit-learn matplotlib` once first.

In [ ]:
# Section 0: run once.
import math
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("pandas", pd.__version__, "  statsmodels", sm.__version__)
print("setup ok")

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/week-11/summit_gear_customers.csv"

df = pd.read_csv(DATA_URL)
print("accounts:", df.shape)
print("missing values:", int(df.isna().sum().sum()), "  duplicate customer_id:", int(df["customer_id"].duplicated().sum()))
print()
print(df.dtypes)

The target is `defaulted`: 1 if the account's receivable was written off as uncollectible
during the period, 0 if it was not written off in the period. Every predictor is measured as of
the start of the period, so the review list can be made then. The
[data dictionary](https://github.com/sean-mccaman/acctg5150-090/blob/main/2026-fall/week-11/summit_gear_customers_dictionary.md) has every column, its unit and its timing.

The **base rate** is the share of accounts where the target is 1, here the share that
defaulted. Calling every account "will not default" is the **flag-nobody benchmark**: it is
right on every account that did not default and catches no one. Every model this week is
compared with that benchmark.

In [ ]:
base_rate = df["defaulted"].mean()
print(f"{int(df['defaulted'].sum()):,} of {len(df):,} accounts defaulted, a base rate of {base_rate:.1%}.")
print(f"The flag-nobody benchmark is right on {1 - base_rate:.1%} of accounts and catches none of the defaults.")

# Exercise 1: Specify and fit (34 points)

**Choose your predictors.** `PREDICTORS` must include `prior_late_count` and `discount_rate`.
You may add any of the other eight Week 10 candidates (`credit_limit`, `avg_balance`, `annual_spend`, `days_since_last_order`, `orders_per_year`, `avg_order_value`, `noise_a`, `noise_b`) and `segment`. Leave out
`days_to_pay`, `industry`, `customer_id`, `tenure_months`, `region` and `defaulted`. The cell after yours checks the list and says why if a column is not allowed.

Adding a column is a choice you will defend in the model definition, so add one only if you
can say why it belongs.

Exercise 1 fits all 2,200 rows because its job is to read the model, and the odds ratios rest
on more data that way. Exercise 2 refits on the training rows alone, so the model can be judged
on rows it has not seen. Choosing predictors after looking at all the rows means the test rows
are not fully unseen; this lab accepts that simplification.

In [ ]:
# Exercise 1: your predictors. Start from the two required columns and add any you can defend.
PREDICTORS = ['prior_late_count', 'discount_rate']

In [ ]:
# @title Check PREDICTORS against this week's rule (run it; you do not edit it)
TABLE_COLUMNS = ['customer_id', 'region', 'segment', 'industry', 'tenure_months', 'credit_limit',
 'avg_balance', 'annual_spend', 'prior_late_count', 'days_since_last_order',
 'orders_per_year', 'discount_rate', 'avg_order_value', 'noise_a', 'noise_b', 'days_to_pay',
 'defaulted']
CANDIDATES = ['credit_limit', 'avg_balance', 'annual_spend', 'prior_late_count', 'days_since_last_order',
 'orders_per_year', 'discount_rate', 'avg_order_value', 'noise_a', 'noise_b']
CATEGORY = 'segment'
ALLOWED = CANDIDATES + [CATEGORY]
REQUIRED = ['prior_late_count', 'discount_rate']
NOT_ALLOWED = {'days_to_pay': 'days_to_pay is known only after an invoice is paid, so it is not '
                "available when the review list is made. It was Week 10's target, not a "
                'predictor here.',
 'industry': 'industry is a relabel of segment: each segment maps to exactly one industry. '
             'Use segment if you want the account type in the model.',
 'customer_id': 'customer_id is an account identifier, not a measurement of the account.',
 'tenure_months': 'tenure_months stays out this week, as in Week 10, so everyone works '
                  'from the same candidate list.',
 'region': 'region stays out this week, as in Week 10, so everyone works from the same '
           'candidate list.',
 'defaulted': 'defaulted is the target. It is known only at the end of the period, so it '
              'cannot predict itself.'}


def validate_predictors(predictors) -> list:
    """Return a list of plain messages, one per problem. An empty list means the list is legal."""
    if not isinstance(predictors, (list, tuple)):
        return ['PREDICTORS has to be a Python list of column names in quotes, like '
                '["prior_late_count", "discount_rate"].']
    problems = []
    seen = set()
    for name in predictors:
        if not isinstance(name, str):
            problems.append(f"{name!r} is not a column name in quotes.")
            continue
        if name in seen:
            problems.append(f"{name} is in the list twice. List each column once.")
            continue
        seen.add(name)
        if name in NOT_ALLOWED:
            problems.append(NOT_ALLOWED[name])
        elif name in TABLE_COLUMNS and name not in ALLOWED:
            problems.append(f"{name} is not on this week's candidate list. The candidates are "
                            "Week 10's ten numeric columns plus segment.")
        elif name not in ALLOWED:
            problems.append(f"{name} is not a column in this table. Check the spelling against "
                            "the data dictionary.")
    missing = [r for r in REQUIRED if r not in seen]
    if missing:
        problems.append("PREDICTORS must include prior_late_count and discount_rate. Add "
                        + " and ".join(missing) + ".")
    return problems


problems = validate_predictors(PREDICTORS)
if problems:
    print("PREDICTORS is not ready yet:")
    for p in problems:
        print("  -", p)
    raise ValueError("Fix PREDICTORS in the cell above, then run that cell and this one again.")
print("PREDICTORS ok:", PREDICTORS)

In [ ]:
# A first look: each numeric predictor's average for accounts that did not default (0) and that defaulted (1).
numeric = [p for p in PREDICTORS if p != "segment"]
print(df.groupby("defaulted")[numeric].mean().round(2).T)
if "segment" in PREDICTORS:
    print()
    print("default rate by segment:")
    print(df.groupby("segment")["defaulted"].mean().round(3))

**Specify.** Fill in the model definition below.

**Write:** Five lines, one each.

- **Target:** the column, and what 1 and 0 mean
- **Predictors and units:** each predictor you chose, and its unit
- **Why each belongs:** one reason per predictor
- **Look-ahead check:** would you have each value at the start of the period, when the review list is made?
- **Benchmark to beat:** the flag-nobody benchmark, and how often it is right

*Graded on (12 points):* All five lines filled. The target says what 1 and 0 mean. Each predictor has its unit and a business reason, not "it was significant". The look-ahead check says why each value is known at the start of the period. The benchmark names the flag-nobody accuracy.

**Model definition (Exercise 1)**

- **Target:** _the column, and what 1 and 0 mean_
- **Predictors and units:** _each predictor you chose, and its unit_
- **Why each belongs:** _one reason per predictor_
- **Look-ahead check:** _would you have each value at the start of the period, when the review list is made?_
- **Benchmark to beat:** _the flag-nobody benchmark, and how often it is right_

**Fit.** Ask the AI for the `statsmodels` logistic regression (`Logit`) of `defaulted` on
`PREDICTORS`, on all 2,200 rows, with a constant. Name the fitted model `logit_full`. If your
list has `segment`, the prompt below turns it into 0 or 1 columns with one segment left out as
the base case.

In [ ]:
# TODO (Exercise 1): paste the AI's fit here.
# A prompt that works: "Write the statsmodels Logit of defaulted on the columns in PREDICTORS,
# using all rows of df, with a constant. Build X with
# pd.get_dummies(df[PREDICTORS], drop_first=True, dtype=float), and print the summary."
logit_full = None   # replace None with the fitted model

In [ ]:
# Validate: the right rows, a model that converged, and the right predictors?
if logit_full is not None and hasattr(logit_full, "mle_retvals"):
    print("rows used:", int(logit_full.nobs), "(should be 2200)")
    print("converged:", bool(logit_full.mle_retvals.get("converged")), "(should be True)")
    print("predictors:", [p for p in logit_full.params.index if p not in ("const", "Intercept")])
    print()
    print("odds ratios, exp(coefficient), one per predictor:")
    print(np.exp(logit_full.params.drop(["const", "Intercept"], errors="ignore")).round(4).to_string())
else:
    print("logit_full is not a statsmodels Logit fit yet. Ask for sm.Logit and paste it above.")

## Helper: `sigmoid`

A logistic model adds up a **score** for each account: the constant plus each coefficient
times the account's value. The score is the **log-odds** of default, which can be any number.
The sigmoid turns it into a probability between 0 and 1.

- **Takes:** `z`, one account's score.
- **Returns:** a float, `1 / (1 + e ** (-z))`. `math.exp(x)` is `e ** x`.
- **Example:** `sigmoid(0)` is `0.5`. A score of 0 means the odds are even.

**Write it yourself** means you type the body of the function. If Colab offers to finish it for
you (gray suggested code as you type), press Esc and keep going. You can ask the tutor to explain
an idea or an error first. The same goes for the other two helpers.

Grading runs it on scores you have not seen. They are ordinary scores, the size a fitted model
produces, so `math.exp` works without special handling.

In [ ]:
def sigmoid(z):
    """Turn a score (the log-odds) into a probability.

    Takes:
      z -- a number: one account's score from the model (its log-odds).
    Returns:
      float -- 1 / (1 + e ** (-z)), a probability between 0 and 1.
               sigmoid(0) is 0.5. math.exp(x) is e ** x.
    """
    return None

In [ ]:
# From one account's columns to a probability: account C-01182.
# Each value times your model's coefficient, added up, is the score. Your sigmoid turns the
# score into a probability. The last two lines should match.
ACCOUNT = "C-01182"
if logit_full is not None and hasattr(logit_full, "mle_retvals"):
    pos = int(np.flatnonzero(df["customer_id"].to_numpy() == ACCOUNT)[0])
    parts = pd.DataFrame({"coefficient": np.asarray(logit_full.params, dtype=float),
                          "account value": np.asarray(logit_full.model.exog[pos], dtype=float)},
                         index=logit_full.model.exog_names)
    parts["coefficient x value"] = parts["coefficient"] * parts["account value"]
    print(parts.round(4).to_string())
    score = float(parts["coefficient x value"].sum())
    print()
    print(f"score (log-odds) for {ACCOUNT}: {score:.4f}")
    p_yours = sigmoid(score)
    print("your sigmoid(score):    ", None if p_yours is None else round(float(p_yours), 4))
    print("the model's probability:", round(float(np.asarray(logit_full.predict())[pos]), 4))
else:
    print("Fit logit_full above first.")

**Interpret.** The **odds** of default are `p / (1 - p)`: a probability of 0.2 is odds of
0.25, or 1 to 4. An **odds ratio** is e raised to the coefficient, `exp(coefficient)`: how much
the odds are multiplied when that predictor goes up by one unit, holding the other predictors
fixed. Above 1 raises the odds of default and below 1 lowers them. It multiplies the odds, not
the probability.

A dollar column's odds ratio prints as 1.0000, because one dollar is a tiny change. Read it per
$1,000 instead, from the unrounded coefficient: for example
`np.exp(1000 * logit_full.params["avg_balance"])`.

**Write:** Three to five sentences.

- Read two odds ratios in credit terms, holding the other predictors fixed. One of them must be `prior_late_count`.
- Give the probability your model assigns to account C-01182, and say what it means for the credit team.

*Graded on (12 points):* Two odds ratios, one of them `prior_late_count`, read as multipliers on the odds with the other predictors held fixed, and matching your output. The probability for C-01182 from your output, read for the credit team.

**Your interpretation (Exercise 1)**

_Replace this line with your answer._

# Exercise 2: From a probability to a call (32 points)

The credit team will use the model on accounts it has not seen, so judge it on rows held out
from the fit. The split is fixed and **stratified**, which keeps the share of defaults about the
same in both parts, so everyone scores the same 440 test rows. Run it as given.

The deck uses a different random split, so your held-out numbers will differ from the slides'
even with the same predictors. That is expected.

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=37, stratify=df["defaulted"])
print("train rows:", len(train), "  test rows:", len(test), "  defaults in test:", int(test["defaulted"].sum()))

Ask the AI to fit the same model on `train` only, as `logit_train`. Then have it predict the
probability of default for every `train` row, as `train_prob`, and every `test` row, as
`test_prob`. The test columns have to be built the same way as the training columns, in the
same order, which is what the reindex step in the prompt does.

In [ ]:
# TODO (Exercise 2): paste the AI's code here.
# A prompt that works: "Fit the statsmodels Logit of defaulted on PREDICTORS using only the train
# DataFrame. Build X_train with sm.add_constant(pd.get_dummies(train[PREDICTORS], drop_first=True,
# dtype=float)). Build X_test the same way from test, then reindex it to the training columns with
# X_test.reindex(columns=X_train.columns, fill_value=0). Then predict the probability of default
# for the train rows and for the test rows."
logit_train = None   # the model, fit on train only
train_prob = None    # its probabilities for the train rows
test_prob = None     # its probabilities for the test rows

In [ ]:
# Validate: fit on the train rows only, and a probability for every train and test row?
if logit_train is not None and hasattr(logit_train, "mle_retvals") and test_prob is not None and train_prob is not None:
    print("rows used:", int(logit_train.nobs), "(should be 1760)")
    print("converged:", bool(logit_train.mle_retvals.get("converged")), "(should be True)")
    p = np.asarray(test_prob, dtype=float)
    print("train probabilities:", len(np.asarray(train_prob)), "(should be 1760)")
    print("test probabilities:", len(p), "(should be 440), from", round(p.min(), 4), "to", round(p.max(), 4))
else:
    print("Fit logit_train, train_prob and test_prob above first.")

A **cutoff** turns a probability into a call: flag the account when its probability is at or
above the cutoff. Start at 0.5. Every call then lands in one of four cells, and the table of
the four counts is the **confusion matrix**:

| | flagged | not flagged |
|---|---|---|
| **defaulted** | true positive (tp): caught | false negative (fn): missed |
| **did not default** | false positive (fp): a review that was not needed | true negative (tn): cleared |

## Helper: `confusion_counts`

Write it yourself.

- **Takes:** `actual` and `predicted`, two sequences of 0s and 1s the same length (lists, numpy arrays, or pandas Series). `actual` is 1 if the account defaulted; `predicted` is 1 if the model flagged it.
- **Returns:** a dict with four whole numbers: `{"tp": ..., "fp": ..., "fn": ..., "tn": ...}`.
- **Example:** `confusion_counts([1, 0, 1, 0], [1, 1, 0, 0])` is `{"tp": 1, "fp": 1, "fn": 1, "tn": 1}`.

Grading runs it on inputs you have not seen.

In [ ]:
def confusion_counts(actual, predicted):
    """Count the four kinds of call a yes or no model makes.

    Takes:
      actual    -- a sequence of 0s and 1s, 1 if the account defaulted
                   (a list, numpy array, or pandas Series).
      predicted -- a sequence of 0s and 1s the same length, 1 if the model flagged
                   the account. True and False count as 1 and 0.
    Returns:
      dict -- {"tp": ..., "fp": ..., "fn": ..., "tn": ...}, four whole numbers:
              tp  flagged, defaulted          fp  flagged, did not default
              fn  not flagged, defaulted      tn  not flagged, did not default
    """
    return None

In [ ]:
# The calls at 0.5 on the test rows, counted with your helper.
y_test = test["defaulted"].to_numpy()
naive_acc = (y_test == 0).mean()   # the flag-nobody benchmark is right on every account that did not default
counts_050 = acc_050 = precision_050 = recall_050 = None
if test_prob is not None:
    calls_050 = (np.asarray(test_prob, dtype=float) >= 0.5).astype(int)
    counts_050 = confusion_counts(y_test, calls_050)
if counts_050 is not None:
    tp, fp, fn, tn = (counts_050[k] for k in ("tp", "fp", "fn", "tn"))
    print(pd.DataFrame([[tp, fn], [fp, tn]], index=["defaulted", "did not default"], columns=["flagged", "not flagged"]))
    acc_050 = (tp + tn) / len(y_test)                     # the share of all calls that were right
    precision_050 = tp / (tp + fp) if tp + fp else None   # of the accounts flagged, the share that defaulted
    recall_050 = tp / (tp + fn)                           # of the accounts that defaulted, the share flagged
    print()
    print("accuracy at 0.5:           ", round(acc_050, 4))
    print("flag-nobody accuracy:      ", round(naive_acc, 4))
    print("precision at 0.5:          ", None if precision_050 is None else round(precision_050, 4))
    print("recall at 0.5:             ", round(recall_050, 4))
else:
    print("Fit test_prob above and write confusion_counts, then run this again.")

**Precision** is the share of flagged accounts that defaulted: how many of the reviews found a
real problem. **Recall** is the share of defaults the model flagged: how many of the losses it
caught.

**Interpret.**

**Write:** Two or three sentences.

- Compare your model's accuracy at 0.5 with the flag-nobody accuracy on the same test rows, with both numbers.
- Say why the two are so close.
- Say what precision and recall show that accuracy does not.

*Graded on (14 points):* Accuracy at 0.5 and the flag-nobody accuracy on the same test rows, with both numbers. Why they are close. What precision and recall show, each defined correctly.

**Your interpretation (Exercise 2)**

_Replace this line with your answer._

# Exercise 3: Choose a cutoff (32 points)

0.5 is a common starting cutoff, but nothing about this problem makes it the right one,
because the two mistakes do not cost the same. The credit team's numbers: a review costs about
$400 of analyst time, and the team reviews every account the model flags. A default the
team does not see coming costs about $2,000 after recoveries. A review is assumed to catch a
defaulter in time, so a flagged default costs only the review. So the total cost of a set of
calls is

`(tp + fp) × $400 + fn × $2,000`

A true negative costs nothing. The **break-even probability** is where a review and no review
are expected to cost the same: p × $2,000 = $400, so p = 0.20.

## Helper: `total_cost`

Write it yourself.

- **Takes:** `counts`, a dict like the one `confusion_counts` returns, and two costs with defaults: `review_cost=400` and `miss_cost=2000`.
- **Returns:** a number, `(tp + fp) * review_cost + fn * miss_cost`.
- **Example:** `total_cost({"tp": 1, "fp": 2, "fn": 3, "tn": 4})` is `7200`.

Grading runs it on inputs you have not seen, with other costs too.

In [ ]:
def total_cost(counts, review_cost=400, miss_cost=2000):
    """What a set of calls costs the credit team, in dollars.

    Takes:
      counts      -- a dict like the one confusion_counts returns: "tp", "fp", "fn", "tn".
      review_cost -- dollars per account reviewed, default 400. Every flagged account
                     (tp and fp) is reviewed.
      miss_cost   -- dollars per default the model did not flag (fn), default 2000.
    Returns:
      number -- (tp + fp) * review_cost + fn * miss_cost. A true negative costs nothing.
    """
    return None

**Choose the cutoff on the training rows, then score it once on the test rows.** If you chose
the cutoff on the test rows and reported its cost on those same rows, the cost would look
better than the model is likely to do on new accounts, because the choice already saw those
outcomes.

The cell below scores every cutoff from 0.05 to 0.95 on the 1,760 training rows with your two
helpers and takes the cheapest. Two cutoffs can tie. When they do, the cell takes the lower one,
and either is accepted. The dashed line marks the 0.20 break-even.

In [ ]:
# Every cutoff from 0.05 to 0.95, scored on the TRAINING rows with your helpers.
CUTOFFS = [round(0.05 * k, 2) for k in range(1, 20)]
y_train = train["defaulted"].to_numpy()
sweep_train = cutoff_train = cost_train_at_cutoff = None
rows = []
if train_prob is not None:
    p = np.asarray(train_prob, dtype=float)
    for cutoff in CUTOFFS:
        counts = confusion_counts(y_train, (p >= cutoff).astype(int))
        cost = None if counts is None else total_cost(counts)
        if cost is None:
            rows = []
            break
        rows.append({"cutoff": cutoff, **counts, "flagged": counts["tp"] + counts["fp"], "total_cost": cost})
if rows:
    sweep_train = pd.DataFrame(rows)
    print(sweep_train.to_string(index=False))
    best = sweep_train.loc[sweep_train["total_cost"].idxmin()]   # if two cutoffs tie, this takes the lower one
    cutoff_train, cost_train_at_cutoff = float(best["cutoff"]), float(best["total_cost"])
    tied = sweep_train.loc[sweep_train["total_cost"] == best["total_cost"], "cutoff"].tolist()
    print()
    print(f"cheapest cutoff on the training rows: {cutoff_train:.2f}, total cost ${cost_train_at_cutoff:,.0f} on 1,760 rows")
    if len(tied) > 1:
        print("tied at that cost:", ", ".join(f"{t:.2f}" for t in tied), "(the lower one is used; either is accepted)")
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 3.6))
        ax.plot(sweep_train["cutoff"], sweep_train["total_cost"], marker="o", color="#890000", label="total cost, training rows")
        ax.axvline(0.20, ls="--", lw=1, color="#4F4F4F", label="break-even, 0.20")
        ax.scatter([cutoff_train], [cost_train_at_cutoff], s=140, facecolors="none", edgecolors="#CC0000",
                   linewidths=2, zorder=3, label="cheapest cutoff")
        ax.set_xlabel("cutoff")
        ax.set_ylabel("total cost on the 1,760 training rows")
        ax.yaxis.set_major_formatter(lambda v, _: f"${v:,.0f}")
        ax.legend(frameon=False)
        plt.show()
    except Exception as e:
        print("(no chart:", type(e).__name__, ")")
else:
    print("Fit train_prob, and write confusion_counts and total_cost, then run this again.")

In [ ]:
# Your cutoff, scored once on the 440 test rows, beside 0.5 and reviewing nobody.
cost_test_at_cutoff = cost_050 = cost_nobody = None
if cutoff_train is not None and counts_050 is not None:
    p = np.asarray(test_prob, dtype=float)
    counts_at_cutoff = confusion_counts(y_test, (p >= cutoff_train).astype(int))
    counts_nobody = {"tp": 0, "fp": 0, "fn": int(y_test.sum()), "tn": int((y_test == 0).sum())}
    cost_test_at_cutoff = total_cost(counts_at_cutoff)
    cost_050 = total_cost(counts_050)
    cost_nobody = total_cost(counts_nobody)
    table = pd.DataFrame(
        [[c["tp"] + c["fp"], c["tp"], c["fn"], cost] for c, cost in
         ((counts_at_cutoff, cost_test_at_cutoff), (counts_050, cost_050), (counts_nobody, cost_nobody))],
        index=[f"your cutoff, {cutoff_train:.2f}", "cutoff 0.5", "review nobody"],
        columns=["flagged", "defaults caught", "defaults missed", "total cost"])
    print(table.to_string(formatters={"total cost": lambda v: f"${v:,.0f}"}))
else:
    print("Run the training-row sweep and the 0.5 calls above first.")

**AUC on the training rows and the test rows.** The **ROC curve** shows every cutoff at once:
the share of defaults flagged against the share of accounts that did not default but were
flagged. **AUC**, the area under that curve, runs from 0 to 1. It is the chance the model gives a
randomly chosen defaulter a higher probability than a randomly chosen account that did not
default (ties count half), so 0.5 is a coin flip and 1.0 would rank every defaulter first.

Ask the AI for both. `roc_auc_score` is already imported.

In [ ]:
# TODO (Exercise 3): paste the AI's code here.
# A prompt that works: "Using roc_auc_score from sklearn.metrics, compute the AUC of
# test_prob against test['defaulted'], and of train_prob against train['defaulted']."
test_auc = None    # AUC on the test rows
train_auc = None   # AUC on the train rows

In [ ]:
print("train AUC:", None if train_auc is None else round(float(train_auc), 4))
print("test AUC: ", None if test_auc is None else round(float(test_auc), 4))

**Decide.**

**Write:** One paragraph to the credit manager, about 150 to 250 words.

- The cutoff you would hand the credit team, and why.
- How it compares with the 0.20 break-even ($400 / $2,000), and why the two can differ.
- What it costs on the test rows against 0.5 and against reviewing nobody.
- What the train and test AUC comparison says. With 88 defaults in the test rows, a gap of several hundredths can come from the draw alone.
- One thing this table cannot see.

*Graded on (14 points):* The cutoff from your training-row sweep, or a neighbor you defend on cost. How it compares with the 0.20 break-even and why the two can differ. Its test cost against 0.5 and against reviewing nobody, from your output. A hedged reading of the train and test AUC. One thing the table cannot see.

**Your decision (Exercise 3)**

_Replace this line with your answer._

# Final check

Run the cell below. You do not edit it. It prints `RESULTS`, the numbers grading reads,
and runs your three helpers on one fixed input each. Use the **Table of Contents** (the list
icon at the top left in Colab) to jump back to an exercise.

In [ ]:
# ===== Final check. Run it. You do not edit this cell. =====
import json
from IPython.display import display, HTML


def final_check():
    """Collect RESULTS, run the three helpers on one fixed input each, and show what is left."""
    g = globals()

    def num(x, places=4):
        try:
            return None if x is None else round(float(x), places)
        except Exception:
            return None

    def whole(x):
        try:
            return None if x is None else int(round(float(x)))
        except Exception:
            return None

    def fitted(model):
        return model is not None and hasattr(model, "mle_retvals") and hasattr(model, "params")

    def terms(model):
        try:
            return [p for p in model.params.index if p not in ("const", "Intercept")]
        except Exception:
            return None

    preds = g.get("PREDICTORS")
    full, part = g.get("logit_full"), g.get("logit_train")
    train_df, test_df = g.get("train"), g.get("test")
    full_terms = terms(full)

    or_prior = None
    if fitted(full):
        try:
            or_prior = num(np.exp(full.params["prior_late_count"]))
        except Exception:
            or_prior = None

    # Exercise 2's values and every cost but cost_train_at_cutoff are on the 440 test rows.
    results = {
        "predictors": list(preds) if isinstance(preds, (list, tuple)) else None,
        "n_train": None if train_df is None else int(len(train_df)),
        "n_test": None if test_df is None else int(len(test_df)),
        "test_defaults": None if test_df is None else int(test_df["defaulted"].sum()),
        "base_rate": num(df["defaulted"].mean()),
        "or_prior_late": or_prior,
        "acc_050": num(g.get("acc_050")),
        "naive_acc": num(g.get("naive_acc")),
        "precision_050": num(g.get("precision_050")),
        "recall_050": num(g.get("recall_050")),
        "cutoff_train": num(g.get("cutoff_train"), 2),
        "cost_train_at_cutoff": whole(g.get("cost_train_at_cutoff")),
        "cost_test_at_cutoff": whole(g.get("cost_test_at_cutoff")),
        "cost_050": whole(g.get("cost_050")),
        "cost_nobody": whole(g.get("cost_nobody")),
        "train_auc": num(g.get("train_auc")),
        "test_auc": num(g.get("test_auc")),
    }

    # The three helpers, on one fixed input each.
    try:
        sig_out = num(sigmoid(2))
    except Exception as e:
        sig_out = "error: " + type(e).__name__
    try:
        got = confusion_counts([1, 0, 1, 1, 0, 0, 1, 0], [1, 1, 0, 1, 0, 0, 0, 0])
        cm_out = None if got is None else {k: whole(got[k]) for k in ("tp", "fp", "fn", "tn")}
    except Exception as e:
        cm_out = "error: " + type(e).__name__
    try:
        cost_out = whole(total_cost({'tp': 10, 'fp': 20, 'fn': 5, 'tn': 65}))
    except Exception as e:
        cost_out = "error: " + type(e).__name__
    helpers = {"sigmoid": sig_out, "confusion_counts": cm_out, "total_cost": cost_out}

    def ok_predictors():
        return isinstance(preds, (list, tuple)) and validate_predictors(preds) == []

    def terms_match(names):
        numeric = [p for p in preds if p != "segment"]
        extra = [n for n in names if n not in numeric]
        want_extra = 3 if "segment" in preds else 0
        return (sorted(n for n in names if n in numeric) == sorted(numeric)
                and len(extra) == want_extra and all("segment" in n for n in extra))

    def ok_full():
        return (fitted(full) and int(full.nobs) == 2200 and bool(full.mle_retvals.get("converged"))
                and terms_match(full_terms))

    def ok_sigmoid():
        return isinstance(sig_out, float) and abs(sig_out - 0.8808) < 0.001

    def ok_split():
        return (results["n_train"] == 1760 and results["n_test"] == 440
                and results["test_defaults"] == 88)

    def ok_train():
        return (fitted(part) and int(part.nobs) == 1760 and bool(part.mle_retvals.get("converged"))
                and sorted(terms(part)) == sorted(full_terms))

    def ok_prob():
        a = np.asarray(g.get("train_prob"), dtype=float)
        b = np.asarray(g.get("test_prob"), dtype=float)
        return (a.shape == (1760,) and b.shape == (440,)
                and bool(((a >= 0) & (a <= 1)).all()) and bool(((b >= 0) & (b <= 1)).all()))

    def ok_confusion():
        return cm_out == {'tp': 2, 'fp': 1, 'fn': 2, 'tn': 3}

    def ok_metrics():
        c = g.get("counts_050")
        return (isinstance(c, dict) and sum(int(c[k]) for k in ("tp", "fp", "fn", "tn")) == 440
                and None not in (results["acc_050"], results["precision_050"], results["recall_050"]))

    def ok_cost():
        return cost_out == 22000

    def ok_sweep():
        s = g.get("sweep_train")
        return (s is not None and len(s) == 19 and int(s[["tp", "fp", "fn", "tn"]].sum(axis=1).iloc[0]) == 1760
                and results["cutoff_train"] is not None and results["cost_train_at_cutoff"] is not None)

    def ok_scored():
        return None not in (results["cost_test_at_cutoff"], results["cost_050"], results["cost_nobody"])

    def ok_auc():
        a, b = results["test_auc"], results["train_auc"]
        return a is not None and b is not None and 0 <= a <= 1 and 0 <= b <= 1

    checks = [
        ("PREDICTORS: a list that follows the rule",                 "Exercise 1", ok_predictors),
        ("logit_full: 2,200 rows, converged, your PREDICTORS",         "Exercise 1", ok_full),
        ("sigmoid: fixed example",                                    "Exercise 1", ok_sigmoid),
        ("train and test: 1,760 and 440 rows, 88 defaults in test",   "Exercise 2", ok_split),
        ("logit_train: 1,760 rows, converged, the same terms",        "Exercise 2", ok_train),
        ("train_prob and test_prob: 1,760 and 440 probabilities",     "Exercise 2", ok_prob),
        ("confusion_counts: fixed example",                           "Exercise 2", ok_confusion),
        ("accuracy, precision and recall at 0.5",                     "Exercise 2", ok_metrics),
        ("total_cost: fixed example",                                 "Exercise 3", ok_cost),
        ("cutoff chosen on the 1,760 training rows",                  "Exercise 3", ok_sweep),
        ("that cutoff scored once on the 440 test rows",              "Exercise 3", ok_scored),
        ("AUC: training rows and test rows",                          "Exercise 3", ok_auc),
    ]
    rows, passed, todo = [], 0, []
    for label, where, test in checks:
        try:
            good = bool(test())
        except Exception:
            good = False
        if good:
            passed += 1
            rows.append("<tr><td style='font-size:18px;color:#176b3a'>&#10003;</td>"
                        "<td style='padding:2px 14px'>{}</td>"
                        "<td style='color:#176b3a'>done</td></tr>".format(label))
        else:
            todo.append("{} ({})".format(label, where))
            rows.append("<tr><td style='font-size:18px;color:#BE0000'>&#10007;</td>"
                        "<td style='padding:2px 14px'>{}</td>"
                        "<td style='color:#BE0000'>not yet, see <b>{}</b></td></tr>".format(label, where))
    color = "#176b3a" if passed == len(checks) else "#BE0000"
    display(HTML(
        "<div style=\"font-family:Arial,Helvetica,sans-serif\">"
        "<h3 style='margin:4px 0;color:{}'>Lab 11 check: {} of {} done</h3>"
        "<table style='border-collapse:collapse;font-size:14px'>{}</table>"
        "<p style='font-size:12.5px;color:#555;margin-top:8px'>A green row means that piece ran "
        "and has the right shape. It does not grade your numbers or your writing. Grading refits "
        "the PREDICTORS you chose, checks RESULTS against those numbers, runs your three helpers on "
        "inputs you have not seen, and reads your four written cells.</p></div>".format(
            color, passed, len(checks), "".join(rows))))
    print("RESULTS =", json.dumps(results))
    print("HELPERS =", json.dumps(helpers))
    print("MODEL_TERMS =", json.dumps(full_terms))
    if todo:
        print("Still to do:", "; ".join(todo))
    return results


RESULTS = final_check()

## Before you submit

1. **Runtime > Restart session and run all**, so every cell shows its output and the Final check reads your latest work.
2. The Final check prints `RESULTS` with no `null` in it, and every row is green.
3. Your four written cells have your words in them, not the placeholder line.
4. **File > Download > Download .ipynb.** Rename it `<uID>_lab_11.ipynb`, with your uID in place of `<uID>`, and upload it to Lab 11 in Canvas.